# Mass Spectrometry Quality Control

This notebook demonstrates how to use the `ms_qc_tools` package to calculate and interpret various quality control metrics for mass spectrometry data.

In [ ]:
from ms_qc_tools.core import MSQualityControl
import pandas as pd
import json

def print_nice(data):
    """Helper function to nicely print dictionaries or other data types."""
    if isinstance(data, dict):
        print(json.dumps(data, indent=4, default=str))
    else:
        print(data)

qc = MSQualityControl(mzxml_filename=mzxml_file, psm_filename=psm_file)

## 1. Basic Metrics

**Metrics:**
- `total_scan_count`: Total number of scans in the run.
- `ms{level}_scan_count`: Number of scans for each MS level (e.g., MS1, MS2).
- `empty_scan_percentage`: Percentage of scans with 0 peaks. High values may indicate instrument issues or spray instability.

In [ ]:
print("Basic Metrics:")
print_nice(qc.metrics.basic())
print("\nEmpty Scan Percentage:")
print_nice(qc.metrics.empty_scan_percentage())

In [ ]:
_ = qc.plots.dynamic_range()

## 2. Chromatography Metrics

**Metrics:**
- `total_run_time`: Total duration of the run (minutes).
- `ms1_scan_frequency`: Scans per second for MS1. Higher frequencies provide better peak sampling.
- `ms1_tic_cv`: Coefficient of variation of the Total Ion Current (TIC). Lower values indicate a more stable baseline.
- `ms1_tic_smoothness`: Mean absolute difference of TIC normalized by mean TIC. Lower values indicate smoother chromatography.

In [ ]:
print("Chromatography Metrics:")
print_nice(qc.metrics.elution())

In [ ]:
_ = qc.plots.intensity_heatmap_by_rt()

In [ ]:
_ = qc.plots.ms1_feature_map()

## 3. TIC & BPI Metrics

**Metrics:**
- `tic_mean`, `tic_median`, `tic_std`: Statistics for Total Ion Current.
- `bpi_mean`, `bpi_median`, `bpi_std`, `bpi_cv`: Statistics for Base Peak Intensity.
- `tic_p{25,50,...}`, `bpi_p{25,50,...}`: Percentiles for TIC and BPI, useful for understanding the distribution of signal intensity.

In [ ]:
print("TIC/BPI Summary (MS1):")
print_nice(qc.metrics.tic_bpi_summary(ms_level=1))
print("\nTIC/BPI Percentiles (MS1):")
print_nice(qc.metrics.tic_bpi_percentiles(ms_level=1))
print("\nTIC Stability (MS1):")
print_nice(qc.metrics.tic_stability(ms_level=1))

In [ ]:
qc.plots.tic()

In [ ]:
qc.plots.bpi()

## 4. MSn Quality Metrics

**Metrics:**
- `msn_scan_count`: Number of MSn scans.
- `msn_scan_frequency`: Scans per second for MSn.
- `msn_peak_count_mean`, etc.: Statistics on the number of peaks (fragments) per scan.
- `msn_empty_scan_pct`: Percentage of MSn scans with no fragments. High values may indicate poor fragmentation or isolation issues.
- `msn_precursor_mz_min/max/range`: Range of precursor m/z values selected for fragmentation.
- `msn_charge_state_distribution`: Distribution of precursor charge states.
- `msn_log_precursor_intensity_peak_count_correlation`: Correlation between precursor intensity and fragment count. A positive correlation is expected.

In [ ]:
print("MS1 Quality Metrics:")
print_nice(qc.metrics.msn_quality(ms_level=1))

In [ ]:
_ = qc.plots.ms1_quality_metrics()

In [ ]:
print("MS2 Quality Metrics:")
print_nice(qc.metrics.msn_quality(ms_level=2))

In [ ]:
_ = qc.plots.charge_distribution()

## 5. Fragmentation Metrics

**Metrics:**
- `avg_fragmentation_efficiency`: TIC / Precursor Intensity. Indicates how efficiently the precursor was fragmented.
- `efficiency_by_charge`: Fragmentation efficiency grouped by charge state.
- `efficiency_by_mz`: Fragmentation efficiency grouped by precursor m/z bins.

In [ ]:
print("Fragmentation Efficiency (MS2):")
print_nice(qc.metrics.fragmentation_efficiency(ms_level=2))

In [ ]:
_ = qc.plots.fragmentation_efficiency()

In [ ]:
qc.plots.fragments_by_charge()

## 6. Timing & Duty Cycle Metrics

**Metrics:**
- `ms1_time_pct`, `ms2_time_pct`: Percentage of time spent on MS1 vs MS2.
- `avg_ms1_scan_time`, `avg_ms2_scan_time`: Average time per scan.
- `ms2_density`: Distribution of MS2 scans over retention time.
- `mean_cycle_time`: Average time between MS1 scans (duty cycle).
- `mean_msn_per_cycle`: Average number of MS2 scans per MS1 cycle.
- `cycle_efficiency`: MSn scans per cycle / cycle time.

In [ ]:
print("MS2 Timing:")
print_nice(qc.metrics.msn_timing(ms_level=2))
print("\nDuty Cycle:")
print_nice(qc.metrics.cycle())

In [ ]:
_ = qc.plots.msn_timing()

In [ ]:
_ = qc.plots.msn_cycle_metrics()

## 7. Peptide Metrics

**Metrics:**
- `unique_peptide_count`, `unique_protein_count`: Number of unique identifications.
- `avg_peptide_length`, etc.: Peptide length statistics.
- `missed_cleavages_count`, `avg_missed_cleavages`: Missed cleavages statistics. High missed cleavages may indicate digestion issues.
- `avg_hydrophobicity`: Average peptide hydrophobicity.
- `modified_peptide_pct`: Percentage of modified peptides.
- `charge_distribution`: Distribution of peptide charge states.
- `peptide_confidence_metrics`: Statistics on peptide confidence scores (e.g., mean, median, distribution).

In [ ]:
print("Peptide Property Metrics:")
print_nice(qc.metrics.peptide())
print("\nPeptide Charge Distribution:")
print_nice(qc.metrics.peptide_charge_distribution())
print("\nPeptide Confidence Metrics:")
print_nice(qc.metrics.peptide_confidence_metrics())

In [ ]:
_ = qc.plots.peptide_length_distribution()

In [ ]:
_ = qc.plots.amino_acid_composition()

In [ ]:
_ = qc.plots.missed_cleavages()

In [ ]:
_ = qc.plots.semitryptic_peptides()

In [ ]:
_ = qc.plots.hydrophobicity_distribution()

In [ ]:
_ = qc.plots.modification_summary()

In [ ]:
_ = qc.plots.peptide_charge_distribution()

## 8. Identification Metrics

**Metrics:**
- `overall_id_rate`: Percentage of MS2 scans identified.
- `id_rate_by_precursor_intensity`: ID rate binned by precursor intensity. Helps identify sensitivity limits.
- `id_rate_by_charge`: ID rate binned by charge state.
- `calibration`: Mean mass error (ppm) per 100 m/z bin. Should be close to 0 and stable across the m/z range.

In [ ]:
print("ID Rate by Metric:")
print_nice(qc.metrics.id_rate_by_metric(ms_level=2))
print("\nCalibration Deltas:")
print_nice(qc.metrics.calibration())

In [ ]:
_ = qc.plots.id_rate_by_metric("precursor_intensity")

In [ ]:
_ = qc.plots.id_rate_by_metric("fragment_count")

In [ ]:
_ = qc.plots.id_rate_by_metric("charge")

In [ ]:
_ = qc.plots.id_rate_by_metric("rt")

In [ ]:
_ = qc.plots.calibration_deltas()

In [ ]:
_ = qc.plots.ms1_feature_map_with_psms()

In [ ]:
if "_QC_" in str(raw_file):
    qc.save_metrics_to_db(db_path=metrics_db,
                          raw_filename=raw_file)
else:
    print(f"Skipping database save: '{raw_file}' does not contain '_QC_'.")

# 